# M1 Lab 2 — First Eval (LLM-as-a-Judge)

Runs Version A (Concise) vs Version B (Narrative) on the Q4 Marketing email, then has a judge model score both against the golden-set criteria in `eval-harness-proof.md`.

**Setup:** put `GOOGLE_API_KEY=...` in a `.env` file in this folder (already gitignored). Deps are installed in the project `.venv` (`google-genai`, `python-dotenv`).

In [10]:
from kaggle_secrets import UserSecretsClient
from openai import OpenAI

user_secrets = UserSecretsClient()
openai_api_key = user_secrets.get_secret("OPENAI_API_KEY")

client = OpenAI(api_key=openai_api_key)

GENERATOR_MODEL = "gpt-5-mini"
JUDGE_MODEL = "gpt-5.4"

print("OpenAI client ready")

OpenAI client ready


In [12]:
SOURCE_DATA = """
Ascend Analytics verified market-intelligence record:

Stripe:
Pricing model: Pay-as-you-go.
Standard domestic card transaction fee: 2.9% + $0.30 per successful transaction.
No standard monthly platform fee.
Custom pricing is available for businesses with large payment volumes or unique business models.

Adyen:
Pricing model: Interchange++ plus a fixed processing fee.
Processing fee: $0.13 per transaction, plus the applicable payment-method fee.
Pricing varies by payment method and region.
Adyen offers custom commercial terms for enterprise customers.

Important:
These facts represent the verified source record supplied for this evaluation.
Do not use information outside this record.
"""

USER_QUESTION = """
Compare the pricing models of Stripe and Adyen.
"""

SYSTEM_A = (
    "You are Ascend IQ, an AI market-intelligence assistant for enterprise product leaders. "
    "Answer using only the verified Ascend Analytics source information provided. "
    "Summarize the comparison in exactly 3 bullet points under 100 words. "
    "Clearly distinguish facts from interpretation. "
    "Do not invent or add information that is not in the source."
)

SYSTEM_B = (
    "You are Ascend IQ, an AI market-intelligence assistant for enterprise product leaders. "
    "Answer using only the verified Ascend Analytics source information provided. "
    "Write a detailed narrative comparison explaining the key pricing differences, "
    "important caveats, and implications for an enterprise evaluating the two providers. "
    "Clearly distinguish facts from interpretation. "
    "Do not invent or add information that is not in the source."
)

INPUT = f"""
SOURCE DATA:
{SOURCE_DATA}

USER QUESTION:
{USER_QUESTION}
"""

In [13]:
def generate(system_prompt: str, user_message: str) -> str:
    response = client.responses.create(
        model=GENERATOR_MODEL,
        instructions=system_prompt,
        input=user_message,
    )
    return response.output_text.strip()


output_a = generate(SYSTEM_A, INPUT)
output_b = generate(SYSTEM_B, INPUT)

print("=== Version A: Executive Brief ===")
print(output_a)

print("\n=== Version B: Detailed Analysis ===")
print(output_b)

=== Version A: Executive Brief ===
- Fact: Stripe uses pay-as-you-go: standard domestic card fee 2.9% + $0.30 per successful transaction, no standard monthly platform fee; custom pricing available for large volumes or unique business models.
- Fact: Adyen uses Interchange++ plus a fixed processing fee: $0.13 per transaction plus applicable payment-method fees; pricing varies by payment method and region; offers custom commercial terms for enterprise customers.
- Interpretation: Stripe’s pricing is a simpler flat percentage+fixed fee for standard cards; Adyen’s is more granular and variable by payment method/region, with both offering enterprise custom terms.

=== Version B: Detailed Analysis ===
Facts (verified source)
- Stripe
  - Pricing model: Pay-as-you-go.
  - Standard domestic card transaction fee: 2.9% + $0.30 per successful transaction.
  - No standard monthly platform fee.
  - Custom pricing available for businesses with large payment volumes or unique business models.

- Adye

In [14]:
GOLDEN_CRITERIA = """
A high-quality Ascend IQ response MUST:

1. Correctly state Stripe's pricing model as pay-as-you-go, with a standard domestic card fee of 2.9% + $0.30 per successful transaction.

2. Correctly state that Stripe has no standard monthly platform fee and that custom pricing is available for large-volume or unique business models.

3. Correctly state Adyen's pricing model as Interchange++ plus a $0.13 processing fee per transaction and applicable payment-method fees.

4. Preserve the caveat that Adyen pricing varies by payment method and region and that custom enterprise commercial terms are available.

5. Clearly distinguish verified facts from interpretation or implications.

6. Make no factual claims that are unsupported by the supplied Ascend Analytics source.

7. Directly answer the user's pricing-comparison question without obscuring the key decision-relevant information with unnecessary detail.

A response is BAD if it contains a material factual error, invents unsupported information, presents an interpretation as a verified fact, or omits source information that could materially change the comparison.

Differences in format or length alone should not determine correctness.
"""


JUDGE_PROMPT = f"""
You are evaluating two candidate responses from Ascend IQ, a premium B2B market-intelligence AI assistant.

Evaluate each candidate independently against the rubric and the supplied source evidence.

RUBRIC:
{GOLDEN_CRITERIA}

SOURCE DATA:
{SOURCE_DATA}

USER QUESTION:
{USER_QUESTION}

CANDIDATE A:
{output_a}

CANDIDATE B:
{output_b}

For each candidate:

1. State which rubric criteria are satisfied.
2. State which criteria are violated or questionable.
3. Identify any unsupported claims or interpretations.
4. Give a final classification of GOOD or BAD.

Then output a final line exactly in this format:
WINNER: A
WINNER: B
or
WINNER: TIE
"""


response = client.responses.create(
    model=JUDGE_MODEL,
    input=JUDGE_PROMPT,
)

verdict = response.output_text.strip()

print(verdict)

Candidate A

1. Satisfied criteria
- 1: Correctly states Stripe is pay-as-you-go and gives 2.9% + $0.30 per successful domestic card transaction.
- 2: Correctly states Stripe has no standard monthly platform fee and that custom pricing is available for large volumes or unique business models.
- 3: Correctly states Adyen uses Interchange++ plus a $0.13 processing fee per transaction and applicable payment-method fees.
- 4: Preserves that Adyen pricing varies by payment method and region and that custom enterprise terms are available.
- 5: Clearly distinguishes facts from interpretation.
- 6: Does not make unsupported factual claims.
- 7: Directly answers the comparison question concisely.

2. Violated or questionable criteria
- None.

3. Unsupported claims or interpretations
- “Stripe’s pricing is a simpler flat percentage+fixed fee for standard cards” is a reasonable interpretation grounded in the source.
- “Adyen’s is more granular and variable by payment method/region” is also a reas